# Topic 17B — Cleaning Text and Category Columns

Many data problems are text problems: extra spaces, inconsistent case, abbreviations, and multiple labels for the same thing.

**Main lesson link:** [Topic 17 — Data Cleaning with pandas](topic17_data_cleaning.ipynb)


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

filename = "data_cleaning_with_pandas/data/students_messy.csv"
possible_paths = [Path(filename), Path("data") / filename, Path("../data") / filename, Path("../../data") / filename]
data_path = next(path for path in possible_paths if path.exists())
df = pd.read_csv(data_path)
print("Loaded:", data_path)
print(df.shape)


---
## 1. Strip Whitespace


In [ ]:
print("Before:")
for name in df[df["full_name"].str.startswith(" ", na=False)]["full_name"].head(5):
    print(repr(name))

df["full_name"] = df["full_name"].str.strip()

print("\nAfter:")
print(df["full_name"].head())


---
## 2. Standardise Case


In [ ]:
print("Raw region examples:")
print(df["region"].value_counts().head(10))

df["region_clean"] = df["region"].str.strip().str.title()

print("\nAfter strip + title:")
print(df["region_clean"].value_counts().head(10))


---
## 3. Map Known Variants

`.replace()` is useful when you know the exact raw labels and the clean labels you want.


In [ ]:
corrections = {
    "Gr. Accra": "Greater Accra",
    "G.Accra": "Greater Accra",
    "Asante": "Ashanti",
    "Eastern Region": "Eastern",
    "Western Region": "Western",
    "U/E": "Upper East",
}

df["region_clean"] = df["region_clean"].replace(corrections)
print(df["region_clean"].value_counts())


---
## 4. Use a Function for Messier Categories


In [ ]:
def standardise_school_type(value):
    if pd.isna(value):
        return value
    cleaned = str(value).strip().lower()
    if cleaned in {"public", "govt", "government", "gov't"}:
        return "Public"
    if cleaned in {"private", "pvt", "priv"}:
        return "Private"
    return "Unknown"

df["school_type_clean"] = df["school_type"].apply(standardise_school_type)
print(df[["school_type", "school_type_clean"]].drop_duplicates().sort_values("school_type_clean"))


---
## 5. Verify the Fix


In [ ]:
print("Clean regions:", sorted(df["region_clean"].dropna().unique()))
print("Clean school types:", sorted(df["school_type_clean"].dropna().unique()))


### Useful Resources

- [pandas user guide: Working with text data](https://pandas.pydata.org/docs/user_guide/text.html)
- [pandas API: `Series.str.strip`](https://pandas.pydata.org/docs/reference/api/pandas.Series.str.strip.html)
- [pandas API: `Series.replace`](https://pandas.pydata.org/docs/reference/api/pandas.Series.replace.html)


---
## Quick Reflection

1. Why do we usually strip spaces before mapping categories?
2. When is `.replace()` enough?
3. When might a custom cleaning function be clearer?
